# 텍스트 토큰화하기
단편 소설 심판 불러오기

In [63]:
import urllib.request

url = ("https://raw.githubusercontent.com/rickiepark/"
       "llm-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("총 문자 개수:", len(raw_text))
print(raw_text[:99])

총 문자 개수: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [64]:
# 텍스트를 토큰 리스트로 분할하기
import re

text = "Hello, world. Is this— a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this—', 'a', 'test', '?']


In [65]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))
# 출력 결과를 보면 모든 단어와 특수 문자가 분리 된 것을 볼 수 있음!!

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


# 토큰을 토큰ID로 변환하기
토큰을 문자열에서 정수 표현으로 바꿔 토큰 ID 생성하기

In [66]:
# 유니크 -> set에 넣기
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print("단어 수",vocab_size)

# 앞에서 50개
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

단어 수 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [67]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    # 텍스트 -> 정수
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # 정수 -> 텍스트
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 구둣점 문자 앞의 공백을 삭제
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [68]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print("인코딩 결과 뽑기: ",ids,"\n")

print("디코딩 결과 뽑기 : ",tokenizer.decode(ids))

# 근데 ? 훈련 세트 기반으로 토크나이저를 구현했다 그러면 훈련 세트에 없는 새로운 텍스트 샘플을 적용하면 ?

인코딩 결과 뽑기:  [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7] 

디코딩 결과 뽑기 :  " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


# 특수 문맥 토큰 추가하기

In [69]:
all_tokens = sorted(list(set(preprocessed)))
# 알지 못하는 단어를 표하는 것과 텍스트 끝을 나타내기 위해 사용된 토큰을 추가하기
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

len(vocab.items()) # 아까보다 2개 느ㄹ어남!

# 뒤에서 부터
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [70]:
# 모드는 단어와 문장의 경계를 대비하는 토크나이저V2

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # 모르는 단어 바꿔주기!
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [71]:
# 토크나이저V2 테스트
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

# text는 모르는 단어와문장 연결 등 수정된 토크나이저가 대응해야할 것들이 많이 존재!
print(text)



Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [72]:
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


#  바이트 페어 인코딩 ( BPE )

In [73]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace.")

# <|endoftext|>는 특수 토큰으로 인식해서 변환하기
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

# (BPE)은 에러 없이 처리함!! 단어를 some, unknown, Place처럼 더 작은 조각으로 쪼개기 때문에
print(integers)

strings = tokenizer.decode(integers)
# 원래 텍스트로 변경해도 문제 없는 것을 볼 수 있음!!
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## 연습문제 알지 못하는 단어에 대한 바이트 페어 인코딩 확인하기

In [74]:
## 연습문제 알지 못하는 단어에 대한 바이트 페어 인코딩 확인하기
ids = tokenizer.encode("Akwirw ier")
print(ids)

for i in ids:
    print(f"{i} -> {tokenizer.decode([i])}")

print(tokenizer.decode(ids))
print("원문과 동일:", tokenizer.decode(ids) == "Akwirw ier")

[33901, 86, 343, 86, 220, 959]
33901 -> Ak
86 -> w
343 -> ir
86 -> w
220 ->  
959 -> ier
Akwirw ier
원문과 동일: True


# 슬라이싱 윈도우로 데이터 샘플링하기

In [75]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 책의 내용 토큰화하기
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]
context_size = 4 # 입력에 포함 시킬 토큰의 수를 의미

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

# 한 토큰 윈도우 슬라이싱된 것을 볼 수 있음!
print(f"x: {x}")
print(f"y:      {y}")

5145
x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [76]:
# 이제 한번에 하나씩 예측하기 한 토큰씩!!
for i in range( 1, context_size+1):
    context = enc_sample[:i] # 0부터 i-1 까지 잘라서 리스트 넣기
    desired = enc_sample[i] # i번째 꺼내기
    # LLM에 들어가는 입력 리스트 --> 예측값 구조임!!!
    print(context , "-->" , desired)

[290] --> 4920
[290, 4920] --> 2241
[290, 4920, 2241] --> 287
[290, 4920, 2241, 287] --> 257


In [77]:
# 이제 한번에 하나씩 예측하기 한 토큰씩!!
for i in range( 1, context_size+1):
    context = enc_sample[:i] # 0부터 i-1 까지 잘라서 리스트 넣기
    desired = enc_sample[i] # i번째 꺼내기
    # LLM에 들어가는 입력 리스트 --> 예측값 구조임!!!
    # 위에서는 토큰 ID 썼고 여기선 텍스트로 출력하기
    print(tokenizer.decode(context) , "-->" , tokenizer.decode([desired]))

 and -->  established
 and established -->  himself
 and established himself -->  in
 and established himself in -->  a


In [78]:
# 토큰을 임베딩으로 바꾸기 전에 입력 데이터 셋을 순회하면서 파이토치 텐서로 입력과 타깃을 반환하는 효율적인 데이터 로더를 구현하기
# 입력 텐서 1개와 예측 타깃을 담은 텐서 1개 만들기

import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"}) # 전체 텍스트 토큰화하기

        for i in range(0, len(token_ids) - max_length, stride):
            input = token_ids[i:i + max_length]
            target = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input))
            self.target_ids.append(torch.tensor(target))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [79]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # 토크나이저를 초기
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋 생성
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터 로더 생성
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [80]:
# context size 4, batch size 1로 데이터 로더 테스트 진행
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# batch_size=1, max_length=4 로 create_dataloader_v1 테스트 해보기!
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

second_batch = next(data_iter)
print(second_batch)


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [81]:
# 연습문제 여러가지 스트라이드와 문맥 크기를 가진 데이터 로더 만들기
# max_length=2와 stride=2 또는 max_length=8와 stride=2 같은 설정으로 테스트 진행


dataloaderPrac1 = create_dataloader_v1(raw_text, batch_size=4, max_length=2, stride=2, shuffle=False)
for i, (x, y) in enumerate(dataloaderPrac1):
    if i >= 3:          # 앞 3개
        break
    print(f"batch {i}: x{tuple(x.shape)}")
    print(x)

batch 0: x(4, 2)
tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619],
        [ 402,  271]])
batch 1: x(4, 2)
tensor([[10899,  2138],
        [  257,  7026],
        [15632,   438],
        [ 2016,   257]])
batch 2: x(4, 2)
tensor([[ 922, 5891],
        [1576,  438],
        [ 568,  340],
        [ 373,  645]])


In [82]:
dataloaderPrac2 = create_dataloader_v1(raw_text, batch_size=4, max_length=8, stride=2, shuffle=False)

for i, (x, y) in enumerate(dataloaderPrac2):
    if i >= 3:          # 앞 3개 배치만
        break
    print(f"batch {i}: x{tuple(x.shape)}")
    print(x)

batch 0: x(4, 8)
tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])
batch 1: x(4, 8)
tensor([[10899,  2138,   257,  7026, 15632,   438,  2016,   257],
        [  257,  7026, 15632,   438,  2016,   257,   922,  5891],
        [15632,   438,  2016,   257,   922,  5891,  1576,   438],
        [ 2016,   257,   922,  5891,  1576,   438,   568,   340]])
batch 2: x(4, 8)
tensor([[ 922, 5891, 1576,  438,  568,  340,  373,  645],
        [1576,  438,  568,  340,  373,  645, 1049, 5975],
        [ 568,  340,  373,  645, 1049, 5975,  284,  502],
        [ 373,  645, 1049, 5975,  284,  502,  284, 3285]])


In [83]:
# 배치 크기를 증가시켜서 테스트하기
dataloader2 = create_dataloader_v1( raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader2)
inputs, targets = next(data_iter)
print("입력: ",inputs,"\n")
print("타깃: ",targets)
# 데이터 셋의 완전 사용 max = 4이고, stride도 4이기 때문에!!!
# 배치 사이에 중첩이 있으면 과대적합이 있을 수 있는데 이렇게 하면 과대적합을 피할 수 있음!
# 켭침이 과적합 위험인가?
''' 같은 토큰이 여러 샘플에 반복 등장하면, 한 에포크 안에서 같은 내용을 여러 번 외움
특히 데이터가 작으면 모델이 일반적인 패턴 대신 이 텍스트 자체를 암기하는 쪽으로 기우며 배치 안에 겹치는 샘플이 섞이면 배치 내 다양성도 떨어지는 일이 발생 '''

입력:  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]) 

타깃:  tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


' 같은 토큰이 여러 샘플에 반복 등장하면, 한 에포크 안에서 같은 내용을 여러 번 외움\n특히 데이터가 작으면 모델이 일반적인 패턴 대신 이 텍스트 자체를 암기하는 쪽으로 기우며 배치 안에 겹치는 샘플이 섞이면 배치 내 다양성도 떨어지는 일이 발생 '

# 토큰 임베딩 만들기

In [84]:
# 토큰 ID를 임베딩 백터로 변환하는 것! 임베딩 백터를 랜덤한 값으로 초기화!

# 토큰 ID를 임베딩 백터로 변환하는 방법 예시
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3


# 랜덤 시드 123 사용하기
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)
# 임베딩 층에 있는 가중치 행렬을 출력함

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [85]:
# 토큰 ID에 적용하여 임베딩 백터를 얻기
# 임베딩 백터 -> 필요한 행 하나 인덱스로 뽑겠다는 것!
print(embedding_layer(torch.tensor(3)))

tensor([-0.4015,  0.9666, -1.1481], grad_fn=<EmbeddingBackward0>)


In [86]:
# 4개의 입력 ID에 모두 적용하기
print(embedding_layer(input_ids))
# 아까 처음에 보았던 6 * 3 행렬에서 각 인덱스에 해당하는 행이 뽑힌 것을 볼 수 있음

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


# 단어 위치 인코딩하기

In [87]:
vocab_size = 50257
output_dim = 256
toekn_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1( raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False )
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("토큰 ID \n", inputs)
print("\n입력 크기:\n", inputs.shape)

token_embeddings = toekn_embedding_layer(inputs)
print(token_embeddings.shape)
# 토큰 ID가 256차원 백터로 임베딩 된 것을 알 수 있음!

토큰 ID 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

입력 크기:
 torch.Size([8, 4])
torch.Size([8, 4, 256])


In [88]:
# GPT 모델의 절대 임베딩 방법에서는 token_embedding_layer와 동일한 임데딩 차원을 가지는 또 다른 임베딩 층을 만듦!
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([4, 256])
torch.Size([8, 4, 256])
